## Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import dask.dataframe as dd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')
import os
from shapely.geometry import base as shapely_geom
import matplotlib.dates as mdates

## Error Analysis on reach-level

### Load and merge datasets (reaches, usgs height, swot height)

In [ ]:
usgs_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/usgs_gage_swot_compatible_all_ver2.parquet'
df_usgs = pd.read_parquet(usgs_path)

swot_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/swotall_qc_filtered_vD_ver2.parquet'
df_swot = pd.read_parquet(swot_path)

# time filtering (not needed later)
# df_swot = df_swot[
#     ~((df_swot['time_str'].dt.month == 11) & 
#       (df_swot['time_str'].dt.year == 2025))
# ].copy()

nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet'
# This might be GeoPandas/regular pandas (not Dask)
df_nlcd = gpd.read_parquet(nlcd_path)

In [ ]:
df_swot.time_str.sort_values()

In [ ]:
#!/usr/bin/env python3
"""
Extract cycle IDs and date ranges from SWOT PIXC URL files.
Identifies missing dates in the dataset.

URL Format: SWOT_L2_HR_PIXC_CCC_PPP_TTTX_YYYYMMDDTHHMMSS_YYYYMMDDTHHMMSS_XXXX_XX.nc
- CCC: Cycle ID (3 digits)
- PPP: Pass number (3 digits)
- TTTX: Tile ID
- YYYYMMDDTHHMMSS: Start/End datetime
"""

import os
import re
from glob import glob
from datetime import datetime, timedelta
from collections import defaultdict

def extract_info_from_url(url):
    """
    Extract cycle ID and date from SWOT PIXC URL.
    Example: SWOT_L2_HR_PIXC_002_343_238L_20230823T081320_20230823T081332_PGD0_01.nc
    """
    # Pattern: SWOT_L2_HR_PIXC_CCC_PPP_TILE_YYYYMMDDTHHMMSS_...
    pattern = r'SWOT_L2_HR_PIXC_(\d{3})_(\d{3})_\w+_(\d{8})T\d{6}_'
    match = re.search(pattern, url)
    
    if match:
        cycle_id = int(match.group(1))
        pass_num = int(match.group(2))
        date_str = match.group(3)
        date = datetime.strptime(date_str, '%Y%m%d').date()
        return cycle_id, pass_num, date
    return None, None, None

def process_url_files(directory):
    """
    Process all URL files in the directory and extract dates.
    """
    all_dates = set()
    cycle_info = defaultdict(lambda: {'dates': set(), 'passes': set()})
    
    # Find all URL files
    url_files = glob(os.path.join(directory, '*.txt'))
    
    print(f"Found {len(url_files)} URL files")
    
    total_urls = 0
    for url_file in sorted(url_files):
        with open(url_file, 'r') as f:
            for line in f:
                url = line.strip()
                if not url:
                    continue
                
                total_urls += 1
                cycle_id, pass_num, date = extract_info_from_url(url)
                
                if date:
                    all_dates.add(date)
                    if cycle_id is not None:
                        cycle_info[cycle_id]['dates'].add(date)
                        cycle_info[cycle_id]['passes'].add(pass_num)
    
    print(f"Processed {total_urls} URLs")
    return all_dates, cycle_info

def find_missing_dates(dates):
    """
    Find missing dates in the date range.
    """
    if not dates:
        return []
    
    sorted_dates = sorted(dates)
    min_date = sorted_dates[0]
    max_date = sorted_dates[-1]
    
    # Generate all dates in range
    all_possible_dates = set()
    current = min_date
    while current <= max_date:
        all_possible_dates.add(current)
        current += timedelta(days=1)
    
    # Find missing
    missing = sorted(all_possible_dates - dates)
    return missing

def format_date_ranges(missing_dates):
    """
    Format missing dates into consecutive ranges for cleaner output.
    """
    if not missing_dates:
        return []
    
    ranges = []
    start = missing_dates[0]
    prev = missing_dates[0]
    
    for date in missing_dates[1:]:
        if (date - prev).days == 1:
            prev = date
        else:
            if start == prev:
                ranges.append(str(start))
            else:
                ranges.append(f"{start} ~ {prev}")
            start = date
            prev = date
    
    # Add last range
    if start == prev:
        ranges.append(str(start))
    else:
        ranges.append(f"{start} ~ {prev}")
    
    return ranges

def main():
    # Update this path to your directory
    directory = '/home/kwanhyuckkim_umass_edu/_4_swot-urban/swot-urban/final/PIXC_url_all'
    
    print("="*70)
    print("SWOT PIXC Cycle ID & Date Range Extraction")
    print("="*70)
    
    all_dates, cycle_info = process_url_files(directory)
    
    # Overall statistics
    if all_dates:
        sorted_dates = sorted(all_dates)
        print(f"\n{'='*70}")
        print(f"[OVERALL DATE RANGE]")
        print(f"{'='*70}")
        print(f"  Start Date      : {sorted_dates[0]}")
        print(f"  End Date        : {sorted_dates[-1]}")
        print(f"  Total Days      : {(sorted_dates[-1] - sorted_dates[0]).days + 1}")
        print(f"  Days with Data  : {len(all_dates)}")
        
        # Find missing dates
        missing = find_missing_dates(all_dates)
        print(f"  Missing Days    : {len(missing)}")
        
        if missing:
            print(f"\n[MISSING DATES - Formatted as Ranges]")
            print("-"*70)
            missing_ranges = format_date_ranges(missing)
            for i, r in enumerate(missing_ranges, 1):
                print(f"  {i:3d}. {r}")
    
    # Per-cycle statistics
    if cycle_info:
        print(f"\n{'='*70}")
        print(f"[PER-CYCLE STATISTICS]")
        print(f"{'='*70}")
        print(f"{'Cycle':<8} {'Start Date':<12} {'End Date':<12} {'Days':<6} {'Missing':<8} {'Passes'}")
        print("-"*70)
        
        for cycle in sorted(cycle_info.keys()):
            info = cycle_info[cycle]
            dates = info['dates']
            passes = info['passes']
            sorted_d = sorted(dates)
            missing_c = find_missing_dates(dates)
            
            print(f"{cycle:03d}      {sorted_d[0]}   {sorted_d[-1]}   {len(dates):<6} {len(missing_c):<8} {len(passes)} passes")
        
        # Detailed missing dates per cycle (optional - can be verbose)
        print(f"\n{'='*70}")
        print(f"[DETAILED MISSING DATES PER CYCLE]")
        print(f"{'='*70}")
        
        for cycle in sorted(cycle_info.keys()):
            dates = cycle_info[cycle]['dates']
            missing_c = find_missing_dates(dates)
            
            if missing_c:
                print(f"\nCycle {cycle:03d} - Missing {len(missing_c)} days:")
                missing_ranges = format_date_ranges(missing_c)
                for r in missing_ranges:
                    print(f"    {r}")

if __name__ == "__main__":
    main()

In [ ]:
df_nlcd_essential = df_nlcd[['reach_id','site_no','nlcd_21_mean', 'nlcd_22_mean'\
                              ,'nlcd_23_mean', 'nlcd_24_mean','fctimp_mean','geometry'
]]

In [ ]:
df_swot['wse_relative_m'] = df_swot.groupby('reach_id')['wse'].transform(
    lambda x: x - x.median()
)

In [ ]:
# 1) Attach reach_id to USGS records via site_no
df_usgs_matched = df_nlcd_essential.merge(
    df_usgs,
    on="site_no",
    how="inner",
)
print(f"✓ USGS with reach_id: {len(df_usgs_matched):,} rows")

# 2) Convert time columns to datetime
df_swot["time_dt"] = pd.to_datetime(df_swot["time_str"])
df_usgs_matched["measurement_dt"] = pd.to_datetime(df_usgs_matched["measurement_dt"])

# 3) Sort by the time key (required for merge_asof)
df_swot_sorted = df_swot.sort_values("time_dt").reset_index(drop=True)
df_usgs_sorted = df_usgs_matched.sort_values("measurement_dt").reset_index(drop=True)

print("Finding nearest USGS measurement for each SWOT overpass (within same reach)...")

# 4) For each SWOT time, find the nearest USGS measurement in the same reach_id
df_matched = pd.merge_asof(
    df_swot_sorted,
    df_usgs_sorted,
    left_on="time_dt",
    right_on="measurement_dt",
    by="reach_id",
    direction="nearest",
    suffixes=("_swot", "_usgs"),
)

# 5) Compute absolute time difference in hours (SWOT vs USGS)
df_matched["time_diff_hours"] = (
    (df_matched["time_dt"] - df_matched["measurement_dt"])
    .abs()
    .dt.total_seconds()
    / 3600.0
)

print(f"✓ SWOT overpasses matched: {len(df_matched):,}")
print(f"  Median |Δt|: {df_matched['time_diff_hours'].median():.2f} h")
print(f"  Max |Δt|:    {df_matched['time_diff_hours'].max():.2f} h")

In [ ]:
# SWOT에만 있고 USGS에 없는 reach_id
swot_reaches = set(df_swot['reach_id'].unique())
usgs_reaches = set(df_usgs_sorted['reach_id'].unique())

only_in_swot = swot_reaches - usgs_reaches
print(f"SWOT reaches: {len(swot_reaches)}")
print(f"USGS reaches: {len(usgs_reaches)}")
print(f"SWOT에만 있는 reach: {len(only_in_swot)}")

# NaN 행 확인
nan_rows = df_matched[df_matched['measurement_dt'].isna()]
print(f"NaN rows: {len(nan_rows)}")
print(f"NaN reaches: {nan_rows['reach_id'].nunique()}")

In [ ]:
df_matched = df_matched[df_matched["time_diff_hours"]<0.3] #not needed later

In [ ]:
df_matched["wse_reach"] = df_matched["wse"]

In [ ]:
df_matched.geometry

In [ ]:
gdf_matched = gpd.GeoDataFrame(df_matched, geometry='geometry', crs='EPSG:4326')

output_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_final_shape.parquet'
gdf_matched.to_parquet(output_path)

df_matched = gpd.read_parquet(output_path)

### Error analysis on reach-level

In [ ]:
df_matched = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_final_shape.parquet')

In [ ]:
df = df_matched.copy()

# Relative comparison (median-normalized)
df_matched['error_relative_m'] = df_matched['wse_relative_m'] - df_matched['wse_usgs_relative_m']

In [ ]:
import pandas as pd
import numpy as np

def extract_statistics_for_text_5pct_with_iqr(df_matched, threshold=0.1832):
    df = df_matched.copy()
    # |Relative WSE Error| 계산
    df['error_relative_m'] = (df['wse_relative_m'] - df['wse_usgs_relative_m']).abs()
    df['isa_pct'] = df['fctimp_mean'] * 100
    df = df.dropna(subset=['isa_pct', 'error_relative_m'])
    
    # 5% 간격 Bin 설정 (원본 Figure 2와 동일)
    custom_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, float('inf')]
    custom_labels = ['0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40', '40-45', '45-50', '50+']
    df['custom_bin'] = pd.cut(df['isa_pct'], bins=custom_bins, labels=custom_labels, include_lowest=True, right=True)

    print("="*70)
    print("ERROR STATISTICS BY 5% ISA CATEGORY (INCLUDING IQR)")
    print("="*70)
    
    # 통계량 계산
    stats = df.groupby('custom_bin')['error_relative_m'].agg(
        N='count',
        Median='median',
        P25=lambda x: np.percentile(x, 25) if len(x) > 0 else np.nan,
        P68=lambda x: np.percentile(x, 68) if len(x) > 0 else np.nan,
        P75=lambda x: np.percentile(x, 75) if len(x) > 0 else np.nan
    ).reset_index()
    
    # IQR (Interquartile Range) 계산 추가
    stats['IQR'] = stats['P75'] - stats['P25']
    
    # 소수점 셋째 자리까지 반올림
    stats[['Median', 'P25', 'P68', 'P75', 'IQR']] = stats[['Median', 'P25', 'P68', 'P75', 'IQR']].round(3)
    print(stats.to_string(index=False))
    
    # 배수 계산 (0-5% 대비 50+%)
    p68_lowest = stats.loc[stats['custom_bin'] == '0-5', 'P68'].values[0]
    p68_highest = stats.loc[stats['custom_bin'] == '50+', 'P68'].values[0]
    if p68_lowest > 0:
        print(f"\nFold change (68th percentile, 0-5% to 50+%): {p68_highest / p68_lowest:.1f}-fold")

# 함수 실행
extract_statistics_for_text_5pct_with_iqr(df_matched)

#### Compare relative WSE directly

In [ ]:
def plot_wse_timeseries_by_isa(
    df,
    use_relative=True,
    nrows=3,
    ncols=4,
    figsize=(16, 10),
    outfile="wse_timeseries_by_isa.png",
):
    """
    Panel plots of WSE time series grouped by impervious surface fraction.

    Each panel corresponds to a rounded fctimp_mean value ("isa_group").
    Within each panel, site-wise time series are shown as lines + points.

    X-axis:
      - ticks at ALL observation times (minor ticks)
      - labels only at one representative date per month (major ticks, formatted as YYYY-MM)

    Parameters
    ----------
    df : DataFrame
        Must contain:
        - 'fctimp_mean'
        - 'time_dt' (SWOT time, datetime-like)
        - 'site_no'
        - 'wse', 'wse_swot_compatible_m'                   (absolute WSE)
        - 'wse_relative_m', 'wse_usgs_relative_m' (relative WSE)
    use_relative : bool
        True  -> relative WSE (median-normalized)
        False -> absolute WSE in EGM2008 mean-tide
    """

    df = df.copy()

    # --------------------------------------------------
    # Choose which WSE columns to use
    # --------------------------------------------------
    if use_relative:
        swot_col = "wse_relative_m"
        gage_col = "wse_usgs_relative_m"
        y_label = "Relative WSE (m, median-normalized)"
        title_suffix = "Relative WSE"
    else:
        swot_col = "wse"
        gage_col = "wse_swot_compatible_m"
        y_label = "WSE (m, EGM2008 mean-tide)"
        title_suffix = "Absolute WSE"

    # Basic filtering and types
    df = df.dropna(subset=["fctimp_mean", "time_dt", swot_col, gage_col])
    df["time_dt"] = pd.to_datetime(df["time_dt"])

    # ISA group (for facet panels)
    df["isa_group"] = df["fctimp_mean"].round(2)

    # --------------------------------------------------
    # Decide which ISA groups to show (panels)
    # --------------------------------------------------
    n_panels = nrows * ncols
    unique_groups = np.sort(df["isa_group"].unique())

    if len(unique_groups) > n_panels:
        # pick evenly spaced ISA groups across full range
        idx = np.linspace(0, len(unique_groups) - 1, n_panels).astype(int)
        selected_groups = unique_groups[idx]
    else:
        selected_groups = unique_groups

    # --------------------------------------------------
    # Figure / axes
    # --------------------------------------------------
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figsize,
        sharex=False,
        sharey=False,
    )
    axes = np.array(axes).reshape(-1)

    # Date formatter for major ticks (month labels)
    month_fmt = mdates.DateFormatter("%Y-%m")

    for ax, isa_val in zip(axes, selected_groups):
        sub = df[df["isa_group"] == isa_val].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        # Sort by site and time
        sub = sub.sort_values(["site_no", "time_dt"])

        # --------------------------------------------------
        # 1) Plot lines + scatter per site (no alpha)
        # --------------------------------------------------
        for site_idx, (site, g) in enumerate(sub.groupby("site_no")):
            lab_g = "Gage" if site_idx == 0 else None
            lab_s = "SWOT" if site_idx == 0 else None

            # Gage
            ax.plot(
                g["time_dt"],
                g[gage_col],
                color="tab:blue",
                linewidth=1.0,
                label=lab_g,
            )
            ax.scatter(
                g["time_dt"],
                g[gage_col],
                color="tab:blue",
                s=12,
            )

            # SWOT
            ax.plot(
                g["time_dt"],
                g[swot_col],
                color="tab:orange",
                linewidth=1.0,
                label=lab_s,
            )
            ax.scatter(
                g["time_dt"],
                g[swot_col],
                color="tab:orange",
                s=12,
            )

        # --------------------------------------------------
        # 2) X-axis ticks & labels
        #    - minor ticks: all observation times
        #    - major ticks: one representative date per month (month label)
        # --------------------------------------------------
        # Minor ticks: all unique observation times
        unique_times = np.sort(sub["time_dt"].unique())
        ax.set_xticks(unique_times, minor=True)

        # Major ticks: month representatives (one per YYYY-MM)
        sub["month_start"] = sub["time_dt"].dt.to_period("M").dt.start_time
        month_positions = np.sort(sub["month_start"].unique())
        ax.set_xticks(month_positions)  # major ticks at month starts
        ax.xaxis.set_major_formatter(month_fmt)

        # Rotate labels 90 degrees (vertical)
        ax.tick_params(axis="x", rotation=90, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

        ax.set_title(
            f"Impervious surface fraction ≈ {isa_val:.2f}",
            fontsize=10,
        )

    # Hide unused panels
    for i in range(len(selected_groups), len(axes)):
        axes[i].set_visible(False)

    # --------------------------------------------------
    # Shared labels & legend
    # --------------------------------------------------
    fig.text(
        0.04, 0.5, y_label,
        va="center", rotation="vertical",
        fontsize=12, fontweight="bold",
    )
    fig.text(
        0.5, 0.03, "Time (SWOT overpass)",
        ha="center",
        fontsize=12, fontweight="bold",
    )

    # Global legend from first visible axis
    visible_axes = [ax for ax in axes if ax.get_visible()]
    if visible_axes:
        handles, labels = visible_axes[0].get_legend_handles_labels()
        fig.legend(
            handles, labels,
            loc="upper center",
            ncol=2,
            frameon=False,
            bbox_to_anchor=(0.5, 0.98),
        )

    fig.suptitle(
        f"{title_suffix} time series by impervious surface fraction",
        fontsize=14, fontweight="bold", y=0.995,
    )

    plt.tight_layout(rect=[0.05, 0.05, 0.98, 0.94])
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    print(f"✓ Saved time series panel: {outfile}")
    plt.show()

In [ ]:
# Relative WSE
plot_wse_timeseries_by_isa(
    df=df_matched,
    use_relative=True,
    nrows=3,
    ncols=4,
    outfile="figures/wse_timeseries_by_isa_relative.png",
)

#### Relative errors on ISA bin with quality flags

In [ ]:
def decode_reach_qb(value):
    """Decode reach_q_b bitwise flags"""
    flags = {
        2: 'classification_qual_suspect',
        4: 'geolocation_qual_suspect',
        8: 'water_fraction_suspect',
        128: 'bright_land',
        1024: 'few_area_observations',
        2048: 'few_wse_observations',
        8192: 'far_range_suspect',
        16384: 'near_range_suspect',
        32768: 'partially_observed',
        262144: 'classification_qual_degraded',
        524288: 'geolocation_qual_degraded',
        4194304: 'lake_flagged',
        33554432: 'below_min_fit_points',
        67108864: 'no_area_observations',
        134217728: 'no_wse_observations',
        268435456: 'no_pixels'
    }
    
    active_flags = []
    for bit_value, flag_name in sorted(flags.items()):
        if value & bit_value:  # bitwise AND
            active_flags.append(flag_name)
    
    return active_flags

df_matched['quality_issues'] = df_matched['reach_q_b'].apply(decode_reach_qb)

In [ ]:
d = df_matched.copy()

swot_col = "wse_relative_m"
gage_col = "wse_usgs_relative_m"
err_name = "error_relative_m"

d[err_name] = (d[swot_col] - d[gage_col]).abs()
spike_mask = d[err_name] >= 0.1832
d_spike = d.loc[spike_mask].copy()

spike_fctimp = d_spike.fctimp_mean.unique()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# FIGURE 2: Boxplot - ISA Bins vs Error (Revised)
# =============================================================================
def create_figure2_isa_vs_error(
    df_matched,
    threshold=0.1832,
    figsize=(14, 9),
    save_path=None,
    dpi=300,
    use_log_scale=True,  # True: log scale, False: linear scale
    ylim_max=5           # Y축 최대값
):
    """
    Figure 2: Boxplot showing error distribution by ISA bins.
     
    ISA bins: 0-5, 5-10, ..., 45-50, 50+ (5% intervals)
    50% 이상은 50+ bin에 포함
     
    Parameters:
    -----------
    use_log_scale : bool
        True: log scale with +1 offset
        False: linear scale (no offset)
    ylim_max : float
        Y축 최대값 (default: 5)
    """
    
    df = df_matched.copy()
    
    # |Relative WSE Error| 계산
    df['error_relative_m'] = (df['wse_relative_m'] - df['wse_usgs_relative_m']).abs()
    
    df['isa_pct'] = df['fctimp_mean'] * 100
    
    max_isa = df['isa_pct'].max()
    
    isa_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
    isa_labels = ['0-5', '5-10', '10-15', '15-20', '20-25', '25-30', 
                  '30-35', '35-40', '40-45', '45-50']
    
    df['isa_bin'] = pd.cut(df['isa_pct'], bins=isa_bins, labels=isa_labels, 
                           include_lowest=True, right=True)
    
    df['isa_bin'] = df['isa_bin'].astype(str)
    df.loc[df['isa_pct'] >= 50, 'isa_bin'] = '50+'
    
    isa_labels_final = isa_labels + ['50+']
    
    df = df.dropna(subset=['isa_bin', 'error_relative_m'])
    df = df[df['isa_bin'] != 'nan']
    
    n_total_50plus = (df['isa_pct'] >= 50).sum()
    
    scale_type = "log" if use_log_scale else "linear"
    print(f"Figure 2: ISA bins 0-50% (5% intervals), 50+ (max: {max_isa:.1f}%)")
    print(f"  Scale: {scale_type}, Y-axis max: {ylim_max}")
    print(f"  Total '50+' bin: {n_total_50plus} samples")
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Offset only for log scale
    offset = 1 if use_log_scale else 0
    
    box_data = []
    box_positions = []
    sample_sizes = []
    p68_values = [] # 각 bin의 68%tile 값을 저장할 리스트
    
    for i, bin_label in enumerate(isa_labels_final):
        subset = df[df['isa_bin'] == bin_label]['error_relative_m'].values
        if len(subset) > 0:
            box_data.append(subset + offset)
            box_positions.append(i)
            sample_sizes.append(len(subset))
            # 68%tile 계산 (offset 포함)
            p68 = np.percentile(subset, 68) + offset
            p68_values.append(p68)
        else:
            # 데이터가 없는 경우 처리 (필요시)
            p68_values.append(np.nan) 
    
    bp = ax.boxplot(
        box_data,
        positions=box_positions,
        widths=0.6,
        whis=[5, 95],
        showfliers=True,
        patch_artist=True,
        medianprops=dict(color='white', linewidth=3),
        boxprops=dict(facecolor='olivedrab', edgecolor='black', linewidth=2),
        whiskerprops=dict(color='black', linewidth=2),
        capprops=dict(color='black', linewidth=2),
        flierprops=dict(marker='o', markersize=4, 
                        markerfacecolor='lightgray', markeredgecolor='lightgray',
                        alpha=0.5)
    )
    
    # 68%tile 표시 (녹색 X)
    # box_positions와 p68_values를 사용하여 scatter plot
    valid_indices = [i for i, val in enumerate(p68_values) if not np.isnan(val)]
    valid_positions = [box_positions[i] for i in valid_indices]
    valid_p68 = [p68_values[i] for i in valid_indices]
    
    ax.scatter(valid_positions, valid_p68, marker='x', s=100, color='black', 
               linewidth=3, zorder=5, label='68th Percentile Error')

    for i, (pos, n) in enumerate(zip(box_positions, sample_sizes)):
        upper_whisker = bp['whiskers'][i * 2 + 1]
        y_top = upper_whisker.get_ydata().max()
        
        ax.text(pos, y_top * 1.08, f'n={n}', 
                fontsize=22, fontweight='bold',
                rotation=90,
                ha='center', va='bottom', color='black')
    
    # Target Error line (주황색 실선)
    threshold_plot = threshold + offset
    ax.axhline(threshold_plot, color='black', linestyle='--', linewidth=3,
               alpha=0.9, label=f'Target Error ({threshold:.4f} m)')
    
    # =========================================================================
    # Scale 설정
    # =========================================================================
    if use_log_scale:
        ax.set_yscale('log')
        ax.set_ylim(offset * 0.9, ylim_max + offset)
        
        def log_formatter(y, pos):
            original = y - offset
            if original < 0.01:
                return ''
            elif original < 0.1:
                return f'{original:.2f}'
            elif original < 1:
                return f'{original:.1f}'
            else:
                return f'{int(original)}'
        
        ax.yaxis.set_major_formatter(FuncFormatter(log_formatter))
        
        # Log scale ticks (ylim_max까지)
        all_ticks = [0, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 40]
        tick_values_orig = [v for v in all_ticks if v <= ylim_max]
        tick_values = [v + offset for v in tick_values_orig]
        ax.set_yticks(tick_values)
    else:
        # Linear scale
        ax.set_ylim(0, ylim_max)
        ax.tick_params(axis='y', labelsize=14)
    
    # X-axis
    xticklabels = isa_labels + ['50-60'] # 50+ 라벨을 50-60으로 변경하신다면 이렇게, 아니면 isa_labels_final 그대로
    # 여기서는 isa_labels_final을 그대로 쓰는게 논리적일 수 있으나, 
    # 원하시는 라벨이 있다면 수정하세요. 일단 isa_labels_final 사용.
    # 만약 '50-60'으로 표기를 원하시면 아래 줄 주석 해제
    # xticklabels = isa_labels + ['50+'] 
    
    ax.set_xticks(range(len(isa_labels_final)))
    ax.set_xticklabels(isa_labels_final, rotation=45, ha='right', 
                       fontsize=22, fontweight='bold')
    ax.set_xlim(-0.5, len(isa_labels_final) - 0.5)
    
    # Labels and title
    ax.set_xlabel('Impervious Surface Area (%)', fontweight='bold', fontsize=22)
    ax.set_ylabel('|Relative WSE Error| (m)', fontweight='bold', fontsize=22) # Label 변경
    ax.set_title('SWOT Relative WSE Error Distribution by Impervious Surface Area',
                 fontweight='bold', fontsize=22, pad=15)
    
    ax.tick_params(axis='y', labelsize=22)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # Legend 설정 (기존 boxplot 요소들은 자동 추가가 안되므로 수동 추가 필요할 수 있음)
    # 하지만 ax.scatter와 ax.axhline에 label을 넣었으므로 legend() 호출 시 이들은 자동 포함됨.
    # boxplot의 median 등을 추가하고 싶다면 커스텀 핸들 필요.
    # 여기서는 요청하신 68%tile과 Target Error 위주로 표시.
    
    # 커스텀 레전드 핸들 생성 (Boxplot 설명 추가를 원할 경우)
    # box_handle = mpatches.Patch(facecolor='steelblue', edgecolor='black', label='IQR (25-75%)')
    # median_handle = Line2D([0], [0], color='darkred', linewidth=3, label='Median')
    
    # 현재 자동 생성되는 핸들 가져오기
    handles, labels = ax.get_legend_handles_labels()
    
    # 필요하다면 순서 조정
    # handles = [handles[1], handles[0]] # 예: Target Error 먼저
    
    ax.legend(handles=handles, loc='upper left', fontsize=22, frameon=True, fancybox=True,
              edgecolor='black', framealpha=0.95, prop={'weight': 'bold', 'size': 22})
    
    ax.grid(True, alpha=0.3, axis='y')
    
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_color('black')
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {save_path}")
        
        pdf_path = save_path.replace('.png', '.pdf')
        fig.savefig(pdf_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {pdf_path}")
    
    return fig

In [ ]:
fig2 = create_figure2_isa_vs_error(df_matched, 
                                    save_path="figures/figure2_linear.png",
                                    use_log_scale=False, 
                                    ylim_max=5)

In [ ]:
# =============================================================================
# FIGURE 3: Quality Flags Heatmap - Error 기반 (X축: ISA %)
# =============================================================================
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

def create_figure3_quality_flags(
    df_matched,
    threshold=0.1832,
    figsize=(16, 10),
    save_path=None,
    dpi=300
):
    """
    Figure 3: Quality flags heatmap colored by error.
    
    - X축: ISA (%) - 정렬된 샘플들
    - Y축: Quality flags
    - 색상: Blue (Low Error) -> White (threshold) -> Red (High Error)
    """
    
    df = df_matched.copy()
    
    # Data preparation
    df['error_relative_m'] = (df['wse_relative_m'] - df['wse_usgs_relative_m']).abs()
    df['isa_pct'] = df['fctimp_mean'] * 100
    
    if 'time_dt' not in df.columns:
        if 'time_str' in df.columns:
            df['time_dt'] = pd.to_datetime(df['time_str'])
    else:
        df['time_dt'] = pd.to_datetime(df['time_dt'])
    
    # Sort by ISA (정렬!)
    df = df.sort_values(['isa_pct']).reset_index(drop=True)
    n_samples = len(df)
    
    print(f"Figure 3: {n_samples} samples")
    print(f"  ISA range: {df['isa_pct'].min():.1f}% ~ {df['isa_pct'].max():.1f}%")
    
    # Decode quality flags
    has_quality_flags = 'reach_q_b' in df.columns
    if has_quality_flags:
        df['classification_qual_suspect'] = (df['reach_q_b'] & 2) > 0
        df['geolocation_qual_suspect'] = (df['reach_q_b'] & 4) > 0
        df['water_fraction_suspect'] = (df['reach_q_b'] & 8) > 0
        df['bright_land'] = (df['reach_q_b'] & 128) > 0
        df['few_area_observations'] = (df['reach_q_b'] & 1024) > 0
        df['few_wse_observations'] = (df['reach_q_b'] & 2048) > 0
        df['far_range_suspect'] = (df['reach_q_b'] & 8192) > 0
        df['near_range_suspect'] = (df['reach_q_b'] & 16384) > 0
        df['partially_observed'] = (df['reach_q_b'] & 32768) > 0
        df['classification_qual_degraded'] = (df['reach_q_b'] & 262144) > 0
        df['geolocation_qual_degraded'] = (df['reach_q_b'] & 524288) > 0
        df['lake_flagged'] = (df['reach_q_b'] & 4194304) > 0
        df['below_min_fit_points'] = (df['reach_q_b'] & 33554432) > 0
        df['no_area_observations'] = (df['reach_q_b'] & 67108864) > 0
        df['no_wse_observations'] = (df['reach_q_b'] & 134217728) > 0
        df['no_pixels'] = (df['reach_q_b'] & 268435456) > 0
        
        flag_names = [
            'classification_qual_suspect', 'geolocation_qual_suspect',
            'water_fraction_suspect', 'bright_land', 'few_area_observations',
            'few_wse_observations', 'far_range_suspect', 'near_range_suspect',
            'partially_observed', 'classification_qual_degraded',
            'geolocation_qual_degraded', 'lake_flagged', 'below_min_fit_points',
            'no_area_observations', 'no_wse_observations', 'no_pixels'
        ]
        
        flag_display_names = [
            'class_suspect', 'geo_suspect',
            'water_frac', 'bright_land', 'few_area',
            'few_wse', 'far_range', 'near_range',
            'partial_obs', 'class_degraded',
            'geo_degraded', 'lake_flag', 'min_fit_pts',
            'no_area', 'no_wse', 'no_pixels'
        ]
    else:
        print("No quality flags found!")
        return None
    
    # Build heatmap matrix
    n_flags = len(flag_names)
    heatmap_data = np.full((n_flags, n_samples), np.nan)
    
    error_values = df['error_relative_m'].values
    
    for flag_idx, flag in enumerate(flag_names):
        mask = df[flag].values
        heatmap_data[flag_idx, mask] = error_values[mask]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Custom colormap: Blue -> White -> Red (diverging at threshold)
    colors = ['#0571b0', '#92c5de', '#f7f7f7', '#f4a582', '#ca0020']
    cmap = LinearSegmentedColormap.from_list('blue_white_red', colors, N=256)
    cmap.set_bad(color='lightgray')  # NaN = lightgray
    
    # Error range
    vmin = 0
    vmax = min(2.0, np.nanmax(error_values))
    
    # TwoSlopeNorm: threshold를 중심으로 발산
    norm = TwoSlopeNorm(vmin=vmin, vcenter=threshold, vmax=vmax)
    
    # Plot heatmap
    im = ax.imshow(
        heatmap_data,
        aspect='auto',
        cmap=cmap,
        norm=norm,
        interpolation='nearest'
    )
    
    # Colorbar
    cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.045)
    cbar.ax.tick_params(labelsize=12)
    
    # Colorbar label
    cbar.ax.set_ylabel('|Relative Error| (m)', fontsize=14, fontweight='bold', rotation=270, labelpad=20)
    
    # Threshold marker on colorbar
    cbar.ax.axhline(threshold, color='black', linewidth=2, linestyle='-')
    
    # Target Error 라벨 (오른쪽)
    cbar.ax.text(-0.3, threshold, f'Target Error = {threshold} m', 
                 fontsize=12, fontweight='bold', va='center', ha='right', rotation=270,
                 transform=cbar.ax.get_yaxis_transform())
    
    # Low/High Error labels (왼쪽)
    cbar.ax.text(-0.3, vmin + (threshold - vmin) * 0.5, 'Low\nError', 
                 fontsize=12, fontweight='bold', va='center', ha='right', color='#0571b0',
                 transform=cbar.ax.get_yaxis_transform())
    cbar.ax.text(-0.3, threshold + (vmax - threshold) * 0.5, 'High\nError', 
                 fontsize=12, fontweight='bold', va='center', ha='right', color='#ca0020',
                 transform=cbar.ax.get_yaxis_transform())
    
    # =========================================================================
    # X-axis: ISA (%) - 정렬된 샘플 기준으로 tick 위치 계산
    # =========================================================================
    isa_values = df['isa_pct'].values
    isa_tick_values = [0, 0.5, 1, 2, 4, 6, 10, 20, 30, 40, 50]  # 5% 추가, 60% 제거
    
    x_ticks = []
    x_labels = []
    
    for isa_target in isa_tick_values:
        idx = np.searchsorted(isa_values, isa_target)
        if idx < n_samples:
            x_ticks.append(idx)
            x_labels.append(str(isa_target))
    
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, fontsize=12, fontweight='bold')
    ax.set_xlabel('ISA (%)', fontsize=14, fontweight='bold')
    
    # Y-axis: Quality flags
    ax.set_ylabel('Quality Flag', fontsize=14, fontweight='bold')
    ax.set_yticks(range(n_flags))
    ax.set_yticklabels(flag_display_names, fontsize=12, fontweight='bold')
    
    # Title
    ax.set_title('Quality Flags vs Error: Can flags predict high error?',
                 fontsize=16, fontweight='bold', pad=15)
    
    # Grid lines between flags
    for i in range(n_flags - 1):
        ax.axhline(i + 0.5, color='white', linewidth=1, alpha=0.8)
    
    # Thick black border
    for spine in ax.spines.values():
        spine.set_linewidth(2)
        spine.set_color('black')
    
    # Statistics
    print("\n[Flag Statistics]")
    print(f"{'Flag':<25} {'Count':>8} {'High Error':>12} {'Rate':>8}")
    print("-" * 55)
    
    for flag_idx, (flag, display) in enumerate(zip(flag_names, flag_display_names)):
        mask = df[flag].values
        n_flagged = mask.sum()
        if n_flagged > 0:
            high_error_rate = (df.loc[mask, 'error_relative_m'] >= threshold).sum() / n_flagged * 100
            print(f"{display:<25} {n_flagged:>8} {int(high_error_rate):>11}% {high_error_rate:>7.1f}%")
    
    overall_rate = (df['error_relative_m'] >= threshold).sum() / len(df) * 100
    print("-" * 55)
    print(f"{'Overall':<25} {len(df):>8} {int(overall_rate):>11}% {overall_rate:>7.1f}%")
    
    plt.tight_layout()
    
    # Save
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"\n✓ Saved: {save_path}")
        
        pdf_path = save_path.replace('.png', '.pdf')
        fig.savefig(pdf_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved: {pdf_path}")
    
    return fig

In [ ]:
fig3 = create_figure3_quality_flags(
    df_matched,
    threshold=0.1832,
    figsize=(16, 10),
    save_path="figures/figure3_quality_heatmap_ver2.png"
)

In [ ]:
# =============================================================================
# FIGURE 3 (revised): Error distribution by quality flag vs. base rate
# =============================================================================
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt


def create_figure3_quality_flags_boxplot(
  df_matched, threshold=0.1832, figsize=(12, 9), save_path=None, dpi=300,
  flag_color='#4C72B0', base_color='#9e9e9e',
):
  df = df_matched.copy()
  df['error_relative_m'] = (df['wse_relative_m'] - df['wse_usgs_relative_m']).abs()

  if 'reach_q_b' not in df.columns:
      print("No quality flags found!"); return None

  bits = {
      'class_suspect': 2, 'geo_suspect': 4, 'water_frac': 8, 'bright_land': 128,
      'few_area': 1024, 'few_wse': 2048, 'far_range': 8192, 'near_range': 16384,
      'partial_obs': 32768, 'class_degraded': 262144, 'geo_degraded': 524288,
      'lake_flag': 4194304, 'min_fit_pts': 33554432, 'no_area': 67108864,
      'no_wse': 134217728, 'no_pixels': 268435456,
  }
  err_all = df['error_relative_m'].values
  base_rate = (err_all >= threshold).mean() * 100

  # per-flag stats
  recs = []
  for disp, bit in bits.items():
      m = (df['reach_q_b'].values & bit) > 0
      n = int(m.sum())
      err = err_all[m] if n > 0 else np.array([])
      hi = (err >= threshold).mean() * 100 if n > 0 else np.nan
      recs.append({'disp': disp, 'n': n, 'hi': hi, 'err': err})
  recs.sort(key=lambda r: r['n'], reverse=True)   # 활성 많은 flag 위로

  rows = [{'disp': 'All observations', 'n': len(df), 'hi': base_rate,
           'err': err_all, 'base': True}] + [{**r, 'base': False} for r in recs]
  n_rows = len(rows)
  ypos = np.arange(n_rows)[::-1]   # base row at top

  fig, ax = plt.subplots(figsize=figsize)

  # boxes (빈 flag는 건너뛰고 주석만)
  bd, bp_pos, bcol = [], [], []
  for y, r in zip(ypos, rows):
      if r['err'].size > 0:
          bd.append(r['err']); bp_pos.append(y)
          bcol.append(base_color if r['base'] else flag_color)
      else:
          ax.text(0.0, y, '   0 raised', va='center', ha='left',
                  fontsize=18, style='italic', color='gray')

  bp = ax.boxplot(bd, positions=bp_pos, vert=False, widths=0.62, patch_artist=True,
                  showfliers=False, medianprops=dict(color='black', linewidth=2.2),
                  whiskerprops=dict(color='black', linewidth=1.2),
                  capprops=dict(color='black', linewidth=1.2),
                  boxprops=dict(edgecolor='black', linewidth=1.2))
  for patch, col in zip(bp['boxes'], bcol):
      patch.set_facecolor(col); patch.set_alpha(0.9)

  # x-range (오른쪽 꼬리 잘라 박스 가독성↑)
  xmax = min(float(np.nanpercentile(err_all, 95)), 1.5)
  ax.set_xlim(0, 3.5)

  # threshold
  ax.axvline(threshold, color='#ca0020', linestyle='--', linewidth=2.2, zorder=5)
  ax.text(threshold, 0.2, f'  Target T = {threshold:.2f} m',
          color='#ca0020', fontsize=18, fontweight='bold', va='bottom', ha='left')

  # base row 구분선
  ax.axhline(ypos[0] - 0.5, color='gray', linestyle=':', linewidth=1.2)

  for y, r in zip(ypos, rows):
      fw = 'bold' if r['base'] else 'normal'
      if r['n'] > 0:                                   # ← n=0이면 출력 안 함
          ax.text(0.80, y, f"n = {r['n']:,}", transform=ax.get_yaxis_transform(),
                  va='center', ha='left', fontsize=18, fontweight=fw, color='black',
                  bbox=dict(boxstyle='round,pad=0.12', facecolor='white',
                            alpha=0.75, edgecolor='none'))
      if not np.isnan(r['hi']):
          ax.text(1.01, y, f"≥T: {r['hi']:.0f}%", transform=ax.get_yaxis_transform(),
                  va='center', ha='left', fontsize=18, fontweight=fw, color='black')

  ax.set_yticks(ypos)
  labs = ax.set_yticklabels([r['disp'] for r in rows], fontsize=18, fontweight='bold')
  labs[0].set_color('#333333')  # baseline label
  ax.set_xlabel('|Relative WSE Error| (m)', fontsize=18, fontweight='bold')
  ax.tick_params(axis='x', labelsize=18)
  ax.set_title('Quality flags vs. WSE error: flagged observations match the base rate',
               fontsize=18, fontweight='bold', pad=12)
  for sp in ax.spines.values():
      sp.set_linewidth(1.6); sp.set_color('black')
  ax.grid(axis='x', linestyle=':', alpha=0.4)

  # 통계 출력
  print(f"\nBase rate (≥T): {base_rate:.1f}%   |  T={threshold} m   |  N={len(df):,}")
  print(f"{'Flag':<16}{'n':>8}{'≥T %':>8}")
  print("-" * 34)
  for r in recs:  
      if r['n'] > 0:
          print(f"{r['disp']:<16}{r['n']:>8,}{r['hi']:>7.1f}%")
      else:
          print(f"{r['disp']:<16}{0:>8}{'  never':>8}")

  plt.tight_layout()
  if save_path:
      fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
      fig.savefig(save_path.replace('.png', '.pdf'), dpi=dpi, bbox_inches='tight', facecolor='white')
      print(f"\n✓ Saved: {save_path}")
  return fig

In [ ]:
Absolute relative WSE error distribution by quality flag. The top box pools all 12,145 observations as the base rate; each box below shows the error distribution of observations for which that flag is raised (box = interquartile range, line = median, whiskers = 1.5×IQR), with the horizontal axis extended past the target accuracy to expose the high-error tail. The dashed line marks the target accuracy T = 0.18 m, and the right margin reports each flag's activation count and the share of flagged observations exceeding T. A diagnostic flag would shift its box past T relative to the base rate; instead every active flag reproduces the base-rate distribution, bright_land is never raised, and the only flag with an elevated error share, few_wse, activates on 0.4% of observations. The quality flags neither separate high-error from low-error observations nor isolate urban degradation.


In [ ]:
fig3 = create_figure3_quality_flags_boxplot(
  df_matched, threshold=0.1832, figsize=(12, 9), save_path="figures/figure3_quality_heatmap_ver3.png", dpi=300,
  flag_color='#4C72B0', base_color='#9e9e9e'
)

In [ ]:
import pandas as pd
import numpy as np

def calculate_flag_tpr_for_text(df_matched, threshold=0.1832):
    df = df_matched.copy()
    
    # 1. Error 및 ISA 계산
    df['error_relative_m'] = (df['wse_relative_m'] - df['wse_usgs_relative_m']).abs()
    df['isa_pct'] = df['fctimp_mean'] * 100
    df = df.dropna(subset=['isa_pct', 'error_relative_m', 'reach_q_b'])
    
    # 2. 필요한 Quality Flags 디코딩
    df['bright_land'] = (df['reach_q_b'] & 128) > 0
    df['class_qual_suspect'] = (df['reach_q_b'] & 2) > 0
    df['class_qual_degraded'] = (df['reach_q_b'] & 262144) > 0
    # Classification quality는 suspect와 degraded를 합쳐서 봅니다.
    df['class_qual_any'] = df['class_qual_suspect'] | df['class_qual_degraded']
    # Any bit set (모든 비트 중 하나라도 켜진 경우 = 0보다 크면)
    df['any_flag'] = df['reach_q_b'] > 0
    
    print("="*60)
    print("1. HIGHEST ISA BIN (>= 50%)")
    print("="*60)
    
    df_high = df[df['isa_pct'] >= 50]
    total_high = len(df_high)
    
    if total_high > 0:
        # 실제 불량 데이터 (Actual Positives)
        actual_degraded = (df_high['error_relative_m'] > threshold).sum()
        exceed_rate = (actual_degraded / total_high) * 100
        
        # True Positives 계산 (에러가 큰데 플래그도 켜진 경우)
        tp_bright_land = ((df_high['error_relative_m'] > threshold) & df_high['bright_land']).sum()
        tpr_bright_land = (tp_bright_land / actual_degraded) * 100 if actual_degraded > 0 else 0
        
        tp_class = ((df_high['error_relative_m'] > threshold) & df_high['class_qual_any']).sum()
        tpr_class = (tp_class / actual_degraded) * 100 if actual_degraded > 0 else 0
        
        tp_any = ((df_high['error_relative_m'] > threshold) & df_high['any_flag']).sum()
        tpr_any = (tp_any / actual_degraded) * 100 if actual_degraded > 0 else 0
        
        print(f"Total observations in >50% bin: {total_high}")
        print(f"Observations exceeding T={threshold}m: {actual_degraded} ({exceed_rate:.1f}%)")
        print(f"TPR of Bright Land flag: {tpr_bright_land:.1f}%")
        print(f"TPR of Classification Quality flag: {tpr_class:.1f}%")
        print(f"TPR of Combined flag (any bit set): {tpr_any:.1f}%")
    else:
        print("No observations found in >= 50% bin.")

    print("\n" + "="*60)
    print("2. LOWEST ISA BIN (<= 5%)")
    print("="*60)
    
    df_low = df[df['isa_pct'] <= 5]
    total_low = len(df_low)
    
    if total_low > 0:
        # 이 구간에서 플래그가 켜진 관측치 (Predicted Positives)
        flagged_low = df_low['any_flag'].sum()
        
        # False Positives (플래그는 켜졌는데 에러는 Threshold 이하인 경우)
        fp_low = ((df_low['error_relative_m'] <= threshold) & df_low['any_flag']).sum()
        
        # 플래그가 켜진 것들 중 False Positive의 비율 (False Discovery Rate)
        fdr_low = (fp_low / flagged_low) * 100 if flagged_low > 0 else 0
        
        print(f"Total observations in <=5% bin: {total_low}")
        print(f"Total flagged observations (any bit set): {flagged_low}")
        print(f"False Positives (Flagged BUT error <= {threshold}m): {fp_low}")
        print(f"% of flagged obs that are FALSE POSITIVES: {fdr_low:.1f}%")
    else:
        print("No observations found in <= 5% bin.")

# 함수 실행 (df_matched가 메모리에 로드되어 있어야 합니다)
calculate_flag_tpr_for_text(df_matched)

### spike_fctimp

In [ ]:
reach_ids = df_matched['reach_id'].unique().tolist()
total = len(reach_ids)

print(f"Total reaches to process: {total}") # 409개가 되어야 함

for i, r_id in enumerate(reach_ids, 1):
    # 1. Update dynamic progress bar
    clear_output(wait=True)
    
    # 해당 reach_id의 데이터 추출
    d = df_matched[df_matched['reach_id'] == r_id].copy()
    
    # 데이터가 비어있지 않은지 확인
    if not d.empty:
        # 파일명 생성을 위해 해당 reach의 fctimp_mean 값 가져오기 (첫 번째 행의 값)
        f = d['fctimp_mean'].iloc[0]
        
        print(f"Processing: {i} / {total} (Reach ID: {r_id}, fctimp: {f:.6f})")

        if 'geometry' in d.columns:
            d.pop('geometry')
        
        # 2. 파일명에 reach_id를 포함시켜 중복 방지 (핵심!)
        # 예: fctimp_mean_0.1234567_reach_7423123.parquet
        save_path = f'/work/pi_kandread_umass_edu/swot-urban/5_fctimp/fctimp_mean_{f:.6f}_reach_{r_id}.parquet'
        d.to_parquet(save_path)

print("Done!")

### Error analysis with quality flags only for spikes

In [ ]:
def plot_spike_analysis_by_isa(
    df,
    spike_threshold=1.0,
    outfile=None,
):
    """
    Analyze all spikes across all ISA fractions in 4 panels
    """
    
    df = df.copy()
    
    if "time_dt" not in df.columns:
        df["time_dt"] = pd.to_datetime(df.get("time_str", df.get("time")))
    
    df["error_relative_m"] = (df["wse_relative_m"] - df["wse_usgs_relative_m"]).abs()
    spikes = df[df["error_relative_m"] >= spike_threshold].copy()
    
    if spikes.empty:
        print(f"No spikes found with threshold >= {spike_threshold}m")
        return
    
    spikes = spikes.sort_values(["fctimp_mean", "time_dt"]).reset_index(drop=True)
    
    if "obs_area_frac" not in spikes.columns:
        spikes["obs_area_frac"] = spikes["area_detct"] / spikes["area_total"]
    
    spikes["xtrk_dist_km"] = spikes["xtrk_dist"].abs() / 1000.0
    
    # Decode 8 quality flags
    spikes["water_fraction_suspect"] = (spikes["reach_q_b"] & 8) > 0
    spikes["classification_qual_suspect"] = (spikes["reach_q_b"] & 2) > 0
    spikes["geolocation_qual_suspect"] = (spikes["reach_q_b"] & 4) > 0
    spikes["far_range_suspect"] = (spikes["reach_q_b"] & 8192) > 0
    spikes["near_range_suspect"] = (spikes["reach_q_b"] & 16384) > 0
    spikes["classification_qual_degraded"] = (spikes["reach_q_b"] & 262144) > 0
    spikes["geolocation_qual_degraded"] = (spikes["reach_q_b"] & 524288) > 0
    spikes["few_wse_observations"] = (spikes["reach_q_b"] & 2048) > 0
    
    spikes["idx"] = np.arange(len(spikes))
    
    unique_isa = sorted(spikes["fctimp_mean"].unique())
    print(unique_isa)
    colors = sns.color_palette("tab10", n_colors=len(unique_isa))
    isa_colors = dict(zip(unique_isa, colors))
    spikes["color"] = spikes["fctimp_mean"].map(isa_colors)
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
    ax1, ax2, ax3, ax4 = axes
    
    # Panel 1: Delta WSE with Greek symbols
    for isa in unique_isa:
        subset = spikes[spikes["fctimp_mean"] == isa]
        ax1.scatter(subset["idx"], subset["error_relative_m"],
                   c=[isa_colors[isa]], s=60, label=f"ISA={isa:.2f}", alpha=0.7)
    
    ax1.axhline(spike_threshold, color="red", linestyle="--", linewidth=1.5, alpha=0.6)
    ax1.text(len(spikes)*0.98, spike_threshold, f' {spike_threshold} m', 
             va='bottom', ha='right', color='red', fontsize=15, fontweight='bold')
    
    ax1.set_ylabel(r"$|\Delta \mathrm{WSE}_{\mathrm{relative}}|$ (m)", fontsize=12, fontweight="bold")
    all_isa = sorted(df["fctimp_mean"].unique())
    missing_isa = [isa for isa in all_isa if isa not in unique_isa]
    missing_str = f" (excluded: {', '.join([f'{x:.2f}' for x in missing_isa])})" if missing_isa else ""
    
    ax1.set_title(f"SWOT Spike Analysis\n"
                 f"Total spikes: {len(spikes)}, ISA groups: {len(unique_isa)}{missing_str}",
                 fontsize=13, fontweight="bold")
    ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: dark_frac & obs_area_frac - connected by ISA
    for isa in unique_isa:
        subset = spikes[spikes["fctimp_mean"] == isa]
        # dark_frac - circle, connected
        ax2.plot(subset["idx"], subset["dark_frac"],
                marker="o", markersize=5, linestyle="-", linewidth=1,
                color=isa_colors[isa], alpha=0.7)
        # obs_area_frac - x, connected
        ax2.plot(subset["idx"], subset["obs_area_frac"],
                marker="x", markersize=6, linestyle="-", linewidth=1,
                color=isa_colors[isa], alpha=0.7)
    
    # Add dummy lines for legend
    ax2.plot([], [], marker="o", linestyle="-", color="gray", label="dark_frac")
    ax2.plot([], [], marker="x", linestyle="-", color="gray", label="obs_area_frac")
    
    ax2.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.4)
    ax2.set_ylabel("Fraction", fontsize=11, fontweight="bold")
    ax2.legend(loc="best", fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: xtrk_dist only, connected
    for isa in unique_isa:
        subset = spikes[spikes["fctimp_mean"] == isa]
        ax3.plot(subset["idx"], subset["xtrk_dist_km"],
                marker="o", markersize=5, linestyle="-", linewidth=1,
                color=isa_colors[isa], alpha=0.7)
    
    ax3.axhline(10, color="red", linestyle="--", linewidth=1.5, alpha=0.6, label="10 km")
    ax3.axhline(60, color="red", linestyle="--", linewidth=1.5, alpha=0.6, label="60 km")
    ax3.set_ylabel("Cross-track distance (km)", fontsize=11, fontweight="bold")
    ax3.legend(loc="best", fontsize=10)
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: 8 quality flags
    flag_names = [
        "water_fraction_suspect",
        "classification_qual_suspect", 
        "geolocation_qual_suspect",
        "few_wse_observations",
        "far_range_suspect",
        "near_range_suspect",
        "classification_qual_degraded",
        "geolocation_qual_degraded"
    ]
    
    markers = ["o", "s", "^", "D", "v", "<", ">", "p"]
    
    for i, (flag, marker) in enumerate(zip(flag_names, markers)):
        mask = spikes[flag]
        if mask.any():
            ax4.scatter(
                spikes.loc[mask, "idx"],
                np.full(mask.sum(), i),
                c=spikes.loc[mask, "color"],
                s=60,
                marker=marker,
                alpha=0.7,
            )
    
    ax4.set_yticks(range(len(flag_names)))
    ax4.set_yticklabels(flag_names, fontsize=9)
    ax4.set_ylabel("Quality flags", fontsize=11, fontweight="bold")
    ax4.set_xlabel("Sequential spike index (sorted by ISA, then time)", fontsize=11, fontweight="bold")
    ax4.grid(True, alpha=0.3, axis='x')
    
    ax4.set_xlim(-1, len(spikes))
    
    # ISA group boundaries
    for ax in axes:
        cumsum = 0
        for isa in unique_isa[:-1]:
            cumsum += (spikes["fctimp_mean"] == isa).sum()
            ax.axvline(cumsum - 0.5, color="gray", linestyle=":", linewidth=1, alpha=0.3)
    
    plt.tight_layout()
    
    if outfile:
        plt.savefig(outfile, dpi=500, bbox_inches="tight")
        print(f"Saved: {outfile}")
    
    plt.show()
    
    # Summary
    print(f"\nSpike Summary:")
    print(f"Total spikes: {len(spikes)}")
    print(f"Unique ISA fractions with spikes: {len(unique_isa)} (out of {df['fctimp_mean'].nunique()} total)")
    for flag in flag_names:
        count = spikes[flag].sum()
        print(f"  {flag}: {count} ({count/len(spikes)*100:.1f}%)")
    
    return spikes

In [ ]:
spikes_df = plot_spike_analysis_by_isa(df_matched, spike_threshold=0.1832, outfile="second_results/spike_analysis.pdf")

## Calculate node-level fctimp_map

In [ ]:
# 1.2 Load SWOT filtered data
print("\nLoading SWOT QC filtered data...")
swot_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_vD_nodeonly_ver2.parquet'
df_swot = pd.read_parquet(swot_path)

In [ ]:
usgs_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/usgs_gage_swot_compatible_all_ver2.parquet'
df_usgs = pd.read_parquet(usgs_path)

swot_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_vD_nodeonly_ver2.parquet'
df_swot = pd.read_parquet(swot_path)

# # 2025 NOV
# df_swot = df_swot[
#     ~((df_swot['time_str'].dt.month == 11) & 
#       (df_swot['time_str'].dt.year == 2025))
# ].copy()

nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet'
# This might be GeoPandas/regular pandas (not Dask)
df_nlcd = gpd.read_parquet(nlcd_path)

In [ ]:
df_nlcd_essential = df_nlcd[['reach_id','site_no','nlcd_21_mean', 'nlcd_22_mean'\
                              ,'nlcd_23_mean', 'nlcd_24_mean','fctimp_mean','geometry'
]]

In [ ]:
df_swot.rename(columns = {"wse":"wse_node"}, inplace = True)
df_swot.columns

In [ ]:
reach_final = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_final_shape.parquet')
reach_time = reach_final.copy()
df_matched = df_swot.merge(
    reach_time, 
    on=['reach_id', 'time_str'], 
    how='inner',
    suffixes=('_node', '_reach')
)
df_matched.columns

In [ ]:
# Create relative WSE for SWOT (median-normalized by reach)
df_matched['wse_reach_relative_m'] = df_matched.groupby('reach_id')['wse_reach'].transform(
    lambda x: x - x.median()
)

df_matched['wse_node_relative_m'] = df_matched.groupby('node_id')['wse_node'].transform(
    lambda x: x - x.median()
)

In [ ]:
df_matched['del_wse'] = df_matched.wse_node_relative_m - df_matched.wse_reach_relative_m
df_matched['error_node'] = df_matched.wse_node_relative_m - df_matched.wse_usgs_relative_m
df_matched['error_reach'] = df_matched.wse_reach_relative_m - df_matched.wse_usgs_relative_m

In [ ]:
node_ids = df_matched['node_id'].unique().tolist()
total = len(node_ids)

print(f"Total nodes to process: {total}") # 409개가 되어야 함

for i, r_id in enumerate(node_ids, 1):
    # 1. Update dynamic progress bar
    clear_output(wait=True)
    
    # 해당 node_id의 데이터 추출
    d = df_matched[df_matched['node_id'] == r_id].copy()
    
    # 데이터가 비어있지 않은지 확인
    if not d.empty:
        # 파일명 생성을 위해 해당 node의 fctimp_mean 값 가져오기 (첫 번째 행의 값)
        f = d['fctimp_mean'].iloc[0]
        
        print(f"Processing: {i} / {total} (Node ID: {r_id}, fctimp: {f:.6f})")

        if 'geometry' in d.columns:
            d.pop('geometry')
        
        # 2. 파일명에 node_id를 포함시켜 중복 방지 (핵심!)
        # 예: fctimp_mean_0.1234567_node_7423123.parquet
        save_path = f'/work/pi_kandread_umass_edu/swot-urban/5_fctimp/fctimp_mean_{f:.6f}_node_{r_id}.parquet'
        d.to_parquet(save_path)

print("Done!")